# BERT

In [5]:
!pip install protobuf

# Or install with transformers dependencies
!pip install transformers[torch]

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.60.1 requires grpcio>=1.60.1, which is not installed.
ray 2.1.0 requires grpcio>=1.32.0; python_version < "3.10", which is not installed.
tensorflow-macos 2.15.0 requires grpcio<2.0,>=1.24.3, which is not installed.
tensorflow-macos 2.15.0 requires h5py>=2.9.0, which is not installed.
tensorflow-macos 2.15.0 requires keras<2.16,>=2.15.0, which is not installed.
tensorflow-macos 2.15.0 requires tensorboard<2.16,>=2.15, which is not installed.
tensorflow-macos 2.15.0 requires tensorflow-estimator<2.16,>=2.15.0, which is not installed.
google-ai-generativelanguage 0.4.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 6.33.6 which is incompatible.
g

In [9]:
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TensorFlow warnings

from transformers import BertTokenizer, BertModel
import torch
import numpy as np

print("="*60)
print("BERT: 'Cat is bad' - DETAILED ANALYSIS")
print("="*60)

# Load model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
model.eval()

sentence = "Cat is bad"
print(f"\nInput: '{sentence}'")

# Tokenize
inputs = tokenizer(sentence, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

print(f"Tokens: {tokens}")
print(f"Token IDs: {inputs['input_ids'].tolist()[0]}")

# Generate embeddings
with torch.no_grad():
    outputs = model(**inputs)

embeddings = outputs.last_hidden_state
print(f"\nOutput shape: {embeddings.shape}")
print(f"  → 1 sentence, {embeddings.shape[1]} tokens, 768 dimensions")

# Extract each token's embedding
print("\n" + "="*60)
print("TOKEN EMBEDDINGS")
print("="*60)

for i, token in enumerate(tokens):
    emb = embeddings[0, i, :]  # [batch=0, position=i, all_dimensions]
    
    print(f"\n{i}. Token: '{token}'")
    print(f"   Shape: {emb.shape}")
    print(f"   First 10 values: {emb[:10].numpy()}")
    print(f"   L2 norm: {torch.norm(emb):.4f}")
    print(f"   Mean: {torch.mean(emb):.4f}")
    print(f"   Std: {torch.std(emb):.4f}")

# Compare "Cat" with "bad"
cat_emb = embeddings[0, 1, :].numpy()  # Position 1
bad_emb = embeddings[0, 3, :].numpy()  # Position 3

# Cosine similarity
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

similarity = cosine_similarity(cat_emb, bad_emb)

print("\n" + "="*60)
print("SIMILARITY ANALYSIS")
print("="*60)
print(f"\nCosine similarity between 'Cat' and 'bad': {similarity:.4f}")
print("(Higher = more similar, range: -1 to 1)")

# Compare with different context
print("\n" + "="*60)
print("CONTEXT SENSITIVITY TEST")
print("="*60)

sentences = [
    "Cat is bad",
    "Cat is good",
    "Cat is sleeping"
]

cat_embeddings = []

for sent in sentences:
    inputs = tokenizer(sent, return_tensors="pt")
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get "Cat" embedding (position 1)
    cat_emb = outputs.last_hidden_state[0, 1, :].numpy()
    cat_embeddings.append(cat_emb)
    
    print(f"\n'{sent}'")
    print(f"  'Cat' embedding norm: {np.linalg.norm(cat_emb):.4f}")
    print(f"  'Cat' embedding mean: {np.mean(cat_emb):.4f}")

# Compare "Cat" across different contexts
print("\n" + "="*60)
print("'Cat' EMBEDDING COMPARISON ACROSS CONTEXTS")
print("="*60)

for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        sim = cosine_similarity(cat_embeddings[i], cat_embeddings[j])
        print(f"\n'{sentences[i]}' vs '{sentences[j]}'")
        print(f"  'Cat' similarity: {sim:.4f}")

print("\n" + "="*60)
print("KEY INSIGHT:")
print("="*60)
print("The same word 'Cat' gets DIFFERENT embeddings based on context!")
print("This is the power of BERT's contextualized representations.")

BERT: 'Cat is bad' - DETAILED ANALYSIS

Input: 'Cat is bad'
Tokens: ['[CLS]', 'cat', 'is', 'bad', '[SEP]']
Token IDs: [101, 4937, 2003, 2919, 102]

Output shape: torch.Size([1, 5, 768])
  → 1 sentence, 5 tokens, 768 dimensions

TOKEN EMBEDDINGS

0. Token: '[CLS]'
   Shape: torch.Size([768])
   First 10 values: [-0.07808859  0.23633376  0.00748989 -0.19930461 -0.21934718 -0.21074782
  0.22678672  0.32977104 -0.0873947  -0.02356342]
   L2 norm: 14.2931
   Mean: -0.0099
   Std: 0.5160

1. Token: 'cat'
   Shape: torch.Size([768])
   First 10 values: [-0.24112292 -0.3909084   0.49417976 -0.5961284   0.42886347 -0.17034085
  0.8402244   0.5922511   0.2468363  -0.08944406]
   L2 norm: 12.5046
   Mean: -0.0083
   Std: 0.4514

2. Token: 'is'
   Shape: torch.Size([768])
   First 10 values: [-0.4049647  -0.19689023 -0.04476955 -0.4561386   0.5344377   0.01362636
 -0.2565227   0.91136676 -0.29432696 -0.3559795 ]
   L2 norm: 13.3442
   Mean: -0.0105
   Std: 0.4817

3. Token: 'bad'
   Shape: torch.S

### Visualization

In [10]:
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

from transformers import BertTokenizer, BertModel
import torch
import numpy as np

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
model.eval()

sentence = "Cat is bad"
inputs = tokenizer(sentence, return_tensors="pt")

# Get attention weights
with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

attentions = outputs.attentions
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

print("="*60)
print("ATTENTION WEIGHTS: 'Cat is bad'")
print("="*60)
print(f"\nTokens: {tokens}")
print(f"Number of layers: {len(attentions)}")
print(f"Number of attention heads per layer: {attentions[0].shape[1]}")

# Show attention from last layer, first head
layer_idx = -1  # Last layer
head_idx = 0    # First head

attn_matrix = attentions[layer_idx][0, head_idx, :, :].numpy()

print(f"\n" + "="*60)
print(f"LAYER 12, HEAD 1 - Attention Weights")
print("="*60)

for i, from_token in enumerate(tokens):
    print(f"\n'{from_token}' pays attention to:")
    for j, to_token in enumerate(tokens):
        weight = attn_matrix[i, j]
        bar = '█' * int(weight * 40)
        print(f"  '{to_token:8s}': {weight:.4f} {bar}")

# Specifically analyze "Cat" attention
print("\n" + "="*60)
print("DETAILED: What does 'Cat' attend to?")
print("="*60)

cat_idx = 1  # "Cat" is at position 1
cat_attention = attn_matrix[cat_idx, :]

print("\n'Cat' attention distribution:")
for j, token in enumerate(tokens):
    weight = cat_attention[j]
    percentage = weight * 100
    bar = '█' * int(weight * 50)
    print(f"  {token:8s}: {percentage:5.1f}% {bar}")

# Find which token "Cat" attends to most
max_attn_idx = np.argmax(cat_attention)
print(f"\n'Cat' attends MOST to: '{tokens[max_attn_idx]}' ({cat_attention[max_attn_idx]:.4f})")

# Analyze "bad" attention
print("\n" + "="*60)
print("DETAILED: What does 'bad' attend to?")
print("="*60)

bad_idx = 3  # "bad" is at position 3
bad_attention = attn_matrix[bad_idx, :]

print("\n'bad' attention distribution:")
for j, token in enumerate(tokens):
    weight = bad_attention[j]
    percentage = weight * 100
    bar = '█' * int(weight * 50)
    print(f"  {token:8s}: {percentage:5.1f}% {bar}")

print("\n" + "="*60)
print("✓ ANALYSIS COMPLETE")
print("="*60)

BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


ATTENTION WEIGHTS: 'Cat is bad'

Tokens: ['[CLS]', 'cat', 'is', 'bad', '[SEP]']
Number of layers: 12
Number of attention heads per layer: 12

LAYER 12, HEAD 1 - Attention Weights

'[CLS]' pays attention to:
  '[CLS]   ': 0.6337 █████████████████████████
  'cat     ': 0.0236 
  'is      ': 0.0332 █
  'bad     ': 0.0196 
  '[SEP]   ': 0.2899 ███████████

'cat' pays attention to:
  '[CLS]   ': 0.0370 █
  'cat     ': 0.0298 █
  'is      ': 0.0077 
  'bad     ': 0.0056 
  '[SEP]   ': 0.9199 ████████████████████████████████████

'is' pays attention to:
  '[CLS]   ': 0.0114 
  'cat     ': 0.0060 
  'is      ': 0.0042 
  'bad     ': 0.0022 
  '[SEP]   ': 0.9762 ███████████████████████████████████████

'bad' pays attention to:
  '[CLS]   ': 0.0145 
  'cat     ': 0.0096 
  'is      ': 0.0117 
  'bad     ': 0.0079 
  '[SEP]   ': 0.9563 ██████████████████████████████████████

'[SEP]' pays attention to:
  '[CLS]   ': 0.0092 
  'cat     ': 0.0079 
  'is      ': 0.0051 
  'bad     ': 0.0037 
  '[SEP]

In [11]:
from transformers import BertTokenizer, BertModel
import torch

print("Testing BERT...")

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

sentence = "Cat is bad"
inputs = tokenizer(sentence, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

embeddings = outputs.last_hidden_state

print(f"✓ SUCCESS!")
print(f"  Input: '{sentence}'")
print(f"  Output shape: {embeddings.shape}")
print(f"  Number of tokens: {embeddings.shape[1]}")
print(f"  Embedding dimension: {embeddings.shape[2]}")
print(f"\n  [CLS] embedding: {embeddings[0, 0, :5]}")
print(f"  'Cat' embedding: {embeddings[0, 1, :5]}")
print(f"  'is' embedding: {embeddings[0, 2, :5]}")
print(f"  'bad' embedding: {embeddings[0, 3, :5]}")
print(f"  [SEP] embedding: {embeddings[0, 4, :5]}")


Testing BERT...
✓ SUCCESS!
  Input: 'Cat is bad'
  Output shape: torch.Size([1, 5, 768])
  Number of tokens: 5
  Embedding dimension: 768

  [CLS] embedding: tensor([-0.0781,  0.2363,  0.0075, -0.1993, -0.2193])
  'Cat' embedding: tensor([-0.2411, -0.3909,  0.4942, -0.5961,  0.4289])
  'is' embedding: tensor([-0.4050, -0.1969, -0.0448, -0.4561,  0.5344])
  'bad' embedding: tensor([ 0.1147,  0.2230, -0.3700, -0.2936, -0.3641])
  [SEP] embedding: tensor([ 0.6679,  0.0461, -0.2326,  0.4141, -0.3516])
